# Nokia Patent Portfolio Analysis — SQL Query Notebook

This notebook consolidates the repository’s BigQuery Standard SQL queries, one query per code cell, with a short note before each query.

**Execution status:** archival/reproducibility notebook. The queries require access to Google BigQuery, `patents-public-data.patents.publications`, and the temporary `nokia_analytics` sandbox tables where noted. They were originally run in the BigQuery console; the saved CSV outputs in `data/` are fingerprinted in `data/source_manifest.csv`. Do not run cells here without the required BigQuery project, credentials, and sandbox tables.

**Terminology:** SQL column aliases such as `owner` refer to applicant/assignee names at filing/publication, not verified current ownership. Grant evidence is not verified current legal status.


## Query files

- `sql/01_owner_names.sql`
- `sql/02_lineage_names.sql`
- `sql/03_filings_by_country.sql`
- `sql/04_check_us_kind_codes.sql`
- `sql/05_owner_names_all_years.sql`
- `sql/06_create_portfolio_30y.sql`
- `sql/07_check_portfolio_30y.sql`
- `sql/08_recount_us_30y.sql`
- `sql/09_family_status.sql`
- `sql/10_in_term_by_office.sql`
- `sql/11_new_families_by_year.sql`
- `sql/12_office_filings_by_family.sql`
- `sql/13_spot_check_sample.sql`
- `sql/14_mean_median_patent_measures.sql`
- `sql/15_rivals_30y.sql`
- `sql/16_codebook_sample.sql`
- `sql/17_codebook_sample_h04l49.sql`
- `sql/18_validation_sample_metadata.sql`
- `sql/19_origin_overlap.sql`
- `sql/20_category_period.sql`
- `sql/21_expiry_curve.sql`
- `sql/22_infinera_optical.sql`
- `sql/23_satcom_actor_names.sql`
- `sql/24_satcom_peers_30y.sql`
- `sql/25_ukraine_satcom_assignees.sql`
- `sql/26_ukraine_office_satcom_filings.sql`
- `sql/27_ukraine_coverage_check.sql`
- `sql/28_ukraine_entity_check.sql`
- `sql/29_echostar_ukraine_cpc.sql`
- `sql/30_spacex_name_counts.sql`
- `sql/31_iceye_recent_examples.sql`
- `sql/32_pool_census.sql`
- `sql/33_pilot_sample.sql`


## `01_owner_names.sql`

Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.


In [ ]:
-- Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.
-- Every owner name attached to granted US patents that mentions Nokia or Alcatel.
SELECT
  a.name AS owner,
  COUNT(DISTINCT p.publication_number) AS granted_patents
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND p.kind_code LIKE 'B%'
  AND (UPPER(a.name) LIKE '%NOKIA%' OR UPPER(a.name) LIKE '%ALCATEL%')
GROUP BY owner
ORDER BY granted_patents DESC;


## `02_lineage_names.sql`

Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.


In [ ]:
-- Superseded by 05_owner_names_all_years.sql: the kind code filter below misses US patents granted before 2001.
-- Owner names for companies absorbed through deals, left out of the first query.
-- \b matches whole words, so TRANSLUCENT is excluded. Alcatel names are already in query 01.
SELECT
  a.name AS owner,
  COUNT(DISTINCT p.publication_number) AS granted_patents
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND p.kind_code LIKE 'B%'
  AND REGEXP_CONTAINS(UPPER(a.name), r'\b(LUCENT|BELL LABS|INFINERA)\b')
  AND NOT REGEXP_CONTAINS(UPPER(a.name), r'ALCATEL')
GROUP BY owner
ORDER BY granted_patents DESC;


## `03_filings_by_country.sql`

Where the Nokia family's patent applications were filed.


In [ ]:
-- Where the Nokia family's patent applications were filed.
-- EP is the European Patent Office, not the EU. Chinese-language names are included for CN records.
-- grant_date may be incomplete outside the US, so check granted_applications before trusting it.
SELECT
  p.country_code,
  COUNT(DISTINCT p.application_number) AS applications,
  COUNT(DISTINCT IF(p.grant_date > 0, p.application_number, NULL)) AS granted_applications,
  COUNT(DISTINCT p.family_id) AS patent_families
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE REGEXP_CONTAINS(
    UPPER(a.name),
    r'\bNOKIA\b|\bALCATEL\b|\bLUCENT TECHNOLOGIES\b|\bINFINERA\b|诺基亚|阿尔卡特|朗讯'
  )
GROUP BY p.country_code
ORDER BY applications DESC;


## `04_check_us_kind_codes.sql`

Checks whether filtering on kind code B dropped older US patents.


In [ ]:
-- Checks whether filtering on kind code B dropped older US patents.
SELECT
  SUBSTR(p.kind_code, 1, 1) AS kind,
  COUNT(DISTINCT p.publication_number) AS patents,
  MIN(p.grant_date) AS first_grant,
  MAX(p.grant_date) AS last_grant
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND REGEXP_CONTAINS(
    UPPER(a.name),
    r'\bNOKIA\b|\bALCATEL\b|\bLUCENT TECHNOLOGIES\b|\bINFINERA\b'
  )
GROUP BY kind
ORDER BY patents DESC;


## `05_owner_names_all_years.sql`

Replaces queries 01 and 02. Kind A covers older US patents, B newer ones and E reissues.


In [ ]:
-- Replaces queries 01 and 02. Kind A covers older US patents, B newer ones and E reissues.
-- Design patents (S) and invention registrations (H) are left out.
-- The search is deliberately broad, and false matches get removed afterwards.
SELECT
  a.name AS owner,
  COUNT(DISTINCT p.publication_number) AS granted_patents,
  MIN(p.grant_date) AS first_grant,
  MAX(p.grant_date) AS last_grant
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'US'
  AND p.grant_date > 0
  AND SUBSTR(p.kind_code, 1, 1) IN ('A', 'B', 'E')
  AND REGEXP_CONTAINS(UPPER(a.name), r'NOKIA|ALCATEL|\bLUCENT\b|BELL LABS|INFINERA')
GROUP BY owner
ORDER BY granted_patents DESC;


## `06_create_portfolio_30y.sql`

Working table: filings since 1 Nov 1996, across all patent offices. Later queries use this instead of the 2.88 TB source.


In [ ]:
-- Working table: filings since 1 Nov 1996, across all patent offices. Later queries use this instead of the 2.88 TB source.
-- This is the exact text I ran. It used about 289 GB once. Sandbox tables expire after 60 days, so rerun it to rebuild the table.
CREATE SCHEMA IF NOT EXISTS nokia_analytics OPTIONS (location = 'US');
CREATE TEMP FUNCTION is_family(n STRING) AS ((REGEXP_CONTAINS(UPPER(n), r'NOKIA|ALCATEL|\bLUCENT\b|INFINERA') OR REGEXP_CONTAINS(n, r'诺基亚|阿尔卡特|朗讯|ノキア|노키아')) AND NOT REGEXP_CONTAINS(UPPER(n), r'NOKIAN|ENOKIAN|AUTOLIV') AND NOT REGEXP_CONTAINS(UPPER(n), r'MAILLEFER|SCANINTER') AND NOT REGEXP_CONTAINS(UPPER(n), r'LUCENT (MEDICAL|SKY|TRANS)') AND NOT REGEXP_CONTAINS(UPPER(n), r'LUCENT (ADVANCED|BIOSCIENCES)') AND NOT REGEXP_CONTAINS(UPPER(n), r'LUCENT OPTICS'));
CREATE OR REPLACE TABLE nokia_analytics.portfolio_30y AS SELECT p.publication_number, p.application_number, p.family_id, p.country_code, p.kind_code, p.priority_date, p.filing_date, p.publication_date, p.grant_date, ARRAY(SELECT DISTINCT a.name FROM UNNEST(p.assignee_harmonized) AS a WHERE is_family(a.name)) AS owners, ARRAY(SELECT i.name FROM UNNEST(p.inventor_harmonized) AS i) AS inventors, ARRAY(SELECT DISTINCT i.country_code FROM UNNEST(p.inventor_harmonized) AS i WHERE i.country_code != '') AS inventor_countries, ARRAY(SELECT c.code FROM UNNEST(p.cpc) AS c) AS cpc_codes, (SELECT t.text FROM UNNEST(p.title_localized) AS t WHERE t.language = 'en' LIMIT 1) AS title_en, (SELECT t.text FROM UNNEST(p.abstract_localized) AS t WHERE t.language = 'en' LIMIT 1) AS abstract_en, ARRAY(SELECT c.publication_number FROM UNNEST(p.citation) AS c WHERE c.publication_number != '') AS cited_publications FROM `patents-public-data.patents.publications` AS p WHERE p.filing_date >= 19961101 AND EXISTS (SELECT 1 FROM UNNEST(p.assignee_harmonized) AS a WHERE is_family(a.name));


## `07_check_portfolio_30y.sql`

Quick check of the working table: size, date range and coverage.


In [ ]:
-- Quick check of the working table: size, date range and coverage.
SELECT
  COUNT(*) AS publications,
  COUNT(DISTINCT application_number) AS applications,
  COUNT(DISTINCT family_id) AS families,
  COUNT(DISTINCT country_code) AS offices,
  MIN(filing_date) AS first_filing,
  MAX(filing_date) AS last_filing,
  COUNTIF(title_en IS NULL) AS without_english_title,
  COUNTIF(abstract_en IS NULL) AS without_english_abstract,
  COUNTIF(ARRAY_LENGTH(cpc_codes) = 0) AS without_cpc,
  COUNTIF(ARRAY_LENGTH(inventors) = 0) AS without_inventors,
  COUNTIF(ARRAY_LENGTH(inventor_countries) = 0) AS without_inventor_country,
  COUNTIF(ARRAY_LENGTH(cited_publications) = 0) AS without_citations
FROM nokia_analytics.portfolio_30y;


## `08_recount_us_30y.sql`

US recount from the working table: granted utility patents filed since 1 Nov 1996, by origin and top owner names.


In [ ]:
-- US recount from the working table: granted utility patents filed since 1 Nov 1996, by origin and top owner names.
-- Origin rows overlap because a publication can have multiple owners. Never sum origin rows to calculate a distinct-publication share.
-- The union row calculates the Alcatel/Lucent share at publication level; saved CSV snapshots require a rerun to include it.
WITH us AS (
    SELECT DISTINCT publication_number, o AS owner
    FROM nokia_analytics.portfolio_30y, UNNEST(owners) AS o
    WHERE country_code = 'US'
      AND grant_date > 0
      AND SUBSTR(kind_code, 1, 1) IN ('A', 'B', 'E')
), c AS (
    SELECT publication_number, owner,
        CASE
            WHEN REGEXP_CONTAINS(UPPER(owner), r'TCL|VACUUM|OPTRONICS|KABEL|CABLE|FIBRES|ESPACE|TELSPACE|MOBILE COMM|RADIOTELEPHONE|SUBMARCOM|SUBMARINE|TRANSP SOLUTION|ALSTHOM|ALSTHOLM|ALTHOM|ALTSHOM|COMMISSARIAT|INRIA|LUCENT KK|^TRT ') THEN 'Needs review'
            WHEN UPPER(owner) LIKE '%INFINERA%' THEN 'Infinera'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'SHANGH[AI]{2} BELL') THEN 'Shanghai Bell'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'S[IE]{2}M[EA]NS') THEN 'Nokia Siemens Networks'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'^LUCENT|ATT CORP LUCENT') THEN 'Lucent'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'ALCATEL|LUCENT') THEN 'Alcatel and Alcatel-Lucent'
            WHEN UPPER(owner) LIKE '%NOKIA%' THEN 'Nokia'
            ELSE 'Other'
        END AS origin
    FROM us
)
SELECT 'origin' AS row_type, origin AS label,
    COUNT(DISTINCT publication_number) AS patents,
    COUNT(DISTINCT owner) AS names,
    CAST(NULL AS FLOAT64) AS share_of_unique_publications
FROM c
GROUP BY origin
UNION ALL
SELECT 'total', 'Unique patents', COUNT(DISTINCT publication_number), COUNT(DISTINCT owner), NULL
FROM c
UNION ALL
SELECT 'total', 'Listings', COUNT(*), COUNT(DISTINCT owner), NULL
FROM c
UNION ALL
SELECT 'union', 'Alcatel and Lucent distinct union',
    COUNT(DISTINCT IF(origin IN ('Alcatel and Alcatel-Lucent', 'Lucent'), publication_number, NULL)),
    COUNT(DISTINCT IF(origin IN ('Alcatel and Alcatel-Lucent', 'Lucent'), owner, NULL)),
    COUNT(DISTINCT IF(origin IN ('Alcatel and Alcatel-Lucent', 'Lucent'), publication_number, NULL)) / COUNT(DISTINCT publication_number)
FROM c
UNION ALL (
    SELECT 'name', owner, COUNT(DISTINCT publication_number), 1, NULL
    FROM c
    GROUP BY owner
    ORDER BY 3 DESC
    LIMIT 15
)
ORDER BY row_type, patents DESC;


## `09_family_status.sql`

Sorts all families by grant evidence, using the working table only. Grant evidence is not current legal status:


In [ ]:
-- Sorts all families by grant evidence, using the working table only. Grant evidence is not current legal status:
-- rights may have lapsed or transferred, and grant_date is incomplete outside the US.
-- Saved CSV snapshots predate this office-aware heuristic and the clarified status labels.
CREATE OR REPLACE TABLE nokia_analytics.family_status AS
WITH pubs AS (
    SELECT family_id, country_code, kind_code, filing_date,
        EXISTS(
            SELECT 1 FROM UNNEST(owners) AS o
            WHERE NOT REGEXP_CONTAINS(UPPER(o), r'TCL|OPTRONICS|KABEL|CABLE|FIBRES|MOBILE COMM|RADIOTELEPHONE|SUBMARCOM|SUBMARINE')
        ) AS core_owner,
        CASE
            WHEN country_code = 'WO' THEN FALSE
            WHEN country_code = 'US' THEN grant_date > 0 AND REGEXP_CONTAINS(kind_code, r'^[ABE]')
            WHEN country_code = 'EP' THEN STARTS_WITH(kind_code, 'B')
            WHEN country_code = 'CN' THEN REGEXP_CONTAINS(kind_code, r'^[BC]')
            WHEN country_code = 'JP' THEN grant_date > 0
            WHEN NOT STARTS_WITH(kind_code, 'S') THEN grant_date > 0
            ELSE FALSE
        END AS grant_evidence
    FROM nokia_analytics.portfolio_30y
)
-- Legacy field names are retained for downstream compatibility; they represent grant evidence, not legal status.
SELECT family_id,
    LOGICAL_OR(core_owner) AS core_owner,
    LOGICAL_OR(grant_evidence) AS ever_granted,
    LOGICAL_OR(grant_evidence AND filing_date >= 20061101) AS granted_in_term,
    LOGICAL_OR(NOT grant_evidence AND country_code != 'WO' AND filing_date >= 20210101) AS recent_application,
    LOGICAL_OR(country_code = 'US' AND grant_evidence AND filing_date >= 20061101) AS us_in_term,
    COUNT(DISTINCT IF(grant_evidence AND filing_date >= 20061101, country_code, NULL)) AS offices_in_term,
    MIN(filing_date) AS first_filing,
    MAX(filing_date) AS last_filing
FROM pubs
GROUP BY family_id;

SELECT CASE
    WHEN NOT core_owner THEN '5 Sold-business names only'
    WHEN granted_in_term THEN '1 Grant evidence, within 20 years'
    WHEN recent_application THEN '2 Recent application, no grant detected'
    WHEN ever_granted THEN '3 Grant evidence, past 20 years'
    ELSE '4 No grant detected, older filing'
END AS status,
    COUNT(*) AS families,
    COUNTIF(us_in_term) AS with_us_in_term,
    ROUND(AVG(offices_in_term), 2) AS avg_offices_in_term
FROM nokia_analytics.family_status
GROUP BY status
ORDER BY status;


## `10_in_term_by_office.sql`

Families with a detected grant within 20 years, by patent office. This is the exact text I ran.


In [ ]:
-- Families with a detected grant within 20 years, by patent office. This is the exact text I ran.
SELECT p.country_code, COUNT(DISTINCT p.family_id) AS families FROM nokia_analytics.portfolio_30y AS p JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term AND p.filing_date >= 20061101 AND p.country_code != 'WO' AND NOT STARTS_WITH(p.kind_code, 'S') AND (p.grant_date > 0 OR REGEXP_CONTAINS(p.kind_code, r'^[BCE]')) GROUP BY p.country_code ORDER BY families DESC LIMIT 15;


## `11_new_families_by_year.sql`

New families per year, using the first filing date in the working table. This is part of the exact query I ran.


In [ ]:
-- New families per year, using the first filing date in the working table. This is part of the exact query I ran.
SELECT DIV(first_filing, 10000) AS yr, COUNT(*) AS new_families FROM nokia_analytics.family_status GROUP BY yr ORDER BY yr;


## `12_office_filings_by_family.sql`

Filings in the four focus offices for every family in the table, found by family rather than owner name, which recovers filings with a missing owner name. This is the exact text I ran, at about 10 GB.


In [ ]:
-- Filings in the four focus offices for every family in the table, found by family rather than owner name, which recovers filings with a missing owner name. This is the exact text I ran, at about 10 GB.
SELECT DIV(p.filing_date, 10000) AS yr, COUNT(DISTINCT IF(p.country_code = 'US', p.application_number, NULL)) AS us, COUNT(DISTINCT IF(p.country_code = 'EP', p.application_number, NULL)) AS ep, COUNT(DISTINCT IF(p.country_code = 'CN', p.application_number, NULL)) AS cn, COUNT(DISTINCT IF(p.country_code = 'JP', p.application_number, NULL)) AS jp FROM `patents-public-data.patents.publications` AS p WHERE p.country_code IN ('US', 'EP', 'CN', 'JP') AND p.filing_date >= 19961101 AND p.family_id IN (SELECT family_id FROM nokia_analytics.family_status) GROUP BY yr ORDER BY yr;


## `13_spot_check_sample.sql`

Repeatable random sample for the register spot-check: up to 20 US, 10 EP, 5 CN and 5 JP granted documents in core families. This is the exact text I ran.


In [ ]:
-- Repeatable random sample for the register spot-check: up to 20 US, 10 EP, 5 CN and 5 JP granted documents in core families. This is the exact text I ran.
WITH p AS (SELECT t.publication_number, t.country_code, t.kind_code, t.filing_date, t.grant_date, t.owners[SAFE_OFFSET(0)] AS owner, ROW_NUMBER() OVER (PARTITION BY t.country_code ORDER BY FARM_FINGERPRINT(CONCAT('spot-check-2026', t.publication_number))) AS rn FROM nokia_analytics.portfolio_30y AS t JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term AND t.country_code IN ('US', 'EP', 'CN', 'JP') AND t.filing_date >= 20061101 AND NOT STARTS_WITH(t.kind_code, 'S') AND (t.grant_date > 0 OR REGEXP_CONTAINS(t.kind_code, r'^[BCE]'))) SELECT publication_number, country_code, kind_code, filing_date, grant_date, owner FROM p WHERE (country_code = 'US' AND rn <= 20) OR (country_code = 'EP' AND rn <= 10) OR (country_code IN ('CN', 'JP') AND rn <= 5) ORDER BY country_code, publication_number;


## `14_mean_median_patent_measures.sql`

Mean, median and spread for time to grant and filings per core family. This is the exact text I ran. Offices include WO, so they overstate national coverage.


In [ ]:
-- Mean, median and spread for time to grant and filings per core family. This is the exact text I ran. Offices include WO, so they overstate national coverage.
WITH g AS (SELECT DISTINCT t.application_number, t.country_code, t.family_id, DATE_DIFF(PARSE_DATE('%Y%m%d', CAST(t.grant_date AS STRING)), PARSE_DATE('%Y%m%d', CAST(t.filing_date AS STRING)), DAY) / 365.25 AS years FROM nokia_analytics.portfolio_30y AS t JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term AND t.filing_date BETWEEN 20061101 AND 20151231 AND t.grant_date > 0 AND t.country_code IN ('US', 'EP', 'CN') AND REGEXP_CONTAINS(t.kind_code, r'^[BC]')), f AS (SELECT family_id, COUNT(DISTINCT application_number) AS apps, COUNT(DISTINCT country_code) AS offices FROM nokia_analytics.portfolio_30y AS t JOIN nokia_analytics.family_status AS s USING (family_id) WHERE s.core_owner AND s.granted_in_term GROUP BY family_id) SELECT CONCAT(country_code, ' years to grant') AS measure, COUNT(*) AS n, ROUND(AVG(years), 2) AS mean, ROUND(APPROX_QUANTILES(years, 100)[OFFSET(50)], 2) AS median, ROUND(APPROX_QUANTILES(years, 100)[OFFSET(10)], 2) AS p10, ROUND(APPROX_QUANTILES(years, 100)[OFFSET(90)], 2) AS p90 FROM g GROUP BY country_code UNION ALL SELECT 'Applications per core family', COUNT(*), ROUND(AVG(apps), 2), APPROX_QUANTILES(apps, 100)[OFFSET(50)], APPROX_QUANTILES(apps, 100)[OFFSET(10)], APPROX_QUANTILES(apps, 100)[OFFSET(90)] FROM f UNION ALL SELECT 'Offices per core family', COUNT(*), ROUND(AVG(offices), 2), APPROX_QUANTILES(offices, 100)[OFFSET(50)], APPROX_QUANTILES(offices, 100)[OFFSET(10)], APPROX_QUANTILES(offices, 100)[OFFSET(90)] FROM f ORDER BY measure;


## `15_rivals_30y.sql`




In [ ]:
CREATE OR REPLACE TABLE nokia_analytics.rivals_30y AS SELECT p.publication_number, p.application_number, p.family_id, p.country_code, p.kind_code, p.priority_date, p.filing_date, p.grant_date, ARRAY(SELECT DISTINCT a.name FROM UNNEST(p.assignee_harmonized) AS a WHERE REGEXP_CONTAINS(UPPER(a.name), r'\bCIENA\b|\bCISCO\b|ACACIA COMM|\bLUXTERA\b|ARISTA NETWORKS|\bHUAWEI\b') OR REGEXP_CONTAINS(a.name, r'华为')) AS owners, ARRAY(SELECT c.code FROM UNNEST(p.cpc) AS c) AS cpc_codes, (SELECT t.text FROM UNNEST(p.title_localized) AS t WHERE t.language = 'en' LIMIT 1) AS title_en FROM `patents-public-data.patents.publications` AS p WHERE p.filing_date >= 19961101 AND EXISTS (SELECT 1 FROM UNNEST(p.assignee_harmonized) AS a WHERE REGEXP_CONTAINS(UPPER(a.name), r'\bCIENA\b|\bCISCO\b|ACACIA COMM|\bLUXTERA\b|ARISTA NETWORKS|\bHUAWEI\b') OR REGEXP_CONTAINS(a.name, r'华为'));


## `16_codebook_sample.sql`

Sampling limits exceed the final company slot quotas to provide spares for replacements.


In [ ]:
-- Sampling limits exceed the final company slot quotas to provide spares for replacements.
-- Saved sample CSVs predate the explicit Huawei match and Other fallback; this edit does not regenerate them.
WITH b AS (
    SELECT 'Nokia' AS co, family_id, publication_number, cpc_codes
    FROM nokia_analytics.portfolio_30y
    WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')
    UNION ALL
    SELECT CASE
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CIENA') THEN 'Ciena'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CISCO|ACACIA|LUXTERA') THEN 'Cisco'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'ARISTA') THEN 'Arista'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'HUAWEI') THEN 'Huawei'
        ELSE 'Other'
    END, family_id, publication_number, cpc_codes
    FROM nokia_analytics.rivals_30y
    WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')
), p AS (
    SELECT co, family_id, publication_number,
        CASE WHEN EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE REGEXP_CONTAINS(c, r'^(H04B10/|H04J14/|H04Q11/00)')) THEN 1 ELSE 0 END AS opt,
        CASE WHEN EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE STARTS_WITH(c, 'H04L45/')) THEN 1 ELSE 0 END AS ipr,
        CASE WHEN EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE REGEXP_CONTAINS(c, r'^(H04L49/|H04L47/)')) THEN 1 ELSE 0 END AS dcs
    FROM b
), l AS (
    SELECT 'OPT' AS pool, co, family_id, MIN(publication_number) AS pub FROM p WHERE opt = 1 GROUP BY co, family_id
    UNION ALL
    SELECT 'IPR', co, family_id, MIN(publication_number) FROM p WHERE ipr = 1 GROUP BY co, family_id
    UNION ALL
    SELECT 'DCS', co, family_id, MIN(publication_number) FROM p WHERE dcs = 1 GROUP BY co, family_id
), r AS (
    SELECT *, COUNT(*) OVER (PARTITION BY pool, co) AS pool_size,
        ROW_NUMBER() OVER (PARTITION BY pool, co ORDER BY FARM_FINGERPRINT(CONCAT('codebook-2026', family_id))) AS rn
    FROM l
)
SELECT pool, co, ANY_VALUE(pool_size) AS pool_size, STRING_AGG(pub, ' ' ORDER BY rn) AS sample
FROM r
WHERE rn <= CASE co WHEN 'Nokia' THEN 56 WHEN 'Huawei' THEN 17 WHEN 'Arista' THEN 17 ELSE 30 END
GROUP BY pool, co
ORDER BY pool, co;


## `17_codebook_sample_h04l49.sql`




In [ ]:
WITH b AS (SELECT 'Nokia' AS co, family_id, publication_number, cpc_codes FROM nokia_analytics.portfolio_30y WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B') UNION ALL SELECT CASE WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CIENA') THEN 'Ciena' WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CISCO|ACACIA|LUXTERA') THEN 'Cisco' WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'ARISTA') THEN 'Arista' ELSE 'Huawei' END, family_id, publication_number, cpc_codes FROM nokia_analytics.rivals_30y WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')), l AS (SELECT co, family_id, MIN(publication_number) AS pub FROM b WHERE EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE STARTS_WITH(c, 'H04L49/')) GROUP BY co, family_id), r AS (SELECT *, COUNT(*) OVER (PARTITION BY co) AS pool_size, ROW_NUMBER() OVER (PARTITION BY co ORDER BY FARM_FINGERPRINT(CONCAT('codebook-2026', family_id))) AS rn FROM l) SELECT co, ANY_VALUE(pool_size) AS pool_size, STRING_AGG(pub, ' ' ORDER BY rn) AS sample FROM r WHERE rn <= CASE co WHEN 'Nokia' THEN 60 ELSE 30 END GROUP BY co ORDER BY co;


## `18_validation_sample_metadata.sql`

Family ID and priority date for the 421 labelled patents, for data/validation/sample_metadata.csv.


In [ ]:
-- Family ID and priority date for the 421 labelled patents, for data/validation/sample_metadata.csv.
-- Run on 1 November 2026 with the full list of publication numbers from data/validation/validation_slots.csv in the IN clause.
-- It returned one concatenated cell, which I split into the CSV. Applicant and current assignee in that CSV come from the saved Google Patents pages.
SELECT STRING_AGG(CONCAT(publication_number,'|',CAST(family_id AS STRING),'|',CAST(priority_date AS STRING),'|',CAST(filing_date AS STRING),'|',ARRAY_TO_STRING(owners,';')),'#') AS x FROM (SELECT publication_number,family_id,priority_date,filing_date,owners FROM nokia_analytics.portfolio_30y UNION ALL SELECT publication_number,family_id,priority_date,filing_date,owners FROM nokia_analytics.rivals_30y) WHERE publication_number IN (/* the 421 publication numbers */);


## `19_origin_overlap.sql`

Quantifies the double-counting in the old 46% figure: how many distinct US granted


In [ ]:
-- Quantifies the double-counting in the old 46% figure: how many distinct US granted
-- publications (filed since 1 Nov 1996) carry both an Alcatel-group and a Lucent owner name.
-- Uses the same origin CASE as 08_recount_us_30y.sql, aggregated per publication.
-- Applicant/assignee names at filing, not verified current ownership.
WITH us AS (
    SELECT DISTINCT publication_number, o AS owner
    FROM nokia_analytics.portfolio_30y, UNNEST(owners) AS o
    WHERE country_code = 'US'
      AND grant_date > 0
      AND SUBSTR(kind_code, 1, 1) IN ('A', 'B', 'E')
), c AS (
    SELECT publication_number,
        CASE
            WHEN REGEXP_CONTAINS(UPPER(owner), r'TCL|VACUUM|OPTRONICS|KABEL|CABLE|FIBRES|ESPACE|TELSPACE|MOBILE COMM|RADIOTELEPHONE|SUBMARCOM|SUBMARINE|TRANSP SOLUTION|ALSTHOM|ALSTHOLM|ALTHOM|ALTSHOM|COMMISSARIAT|INRIA|LUCENT KK|^TRT ') THEN 'Needs review'
            WHEN UPPER(owner) LIKE '%INFINERA%' THEN 'Infinera'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'SHANGH[AI]{2} BELL') THEN 'Shanghai Bell'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'S[IE]{2}M[EA]NS') THEN 'Nokia Siemens Networks'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'^LUCENT|ATT CORP LUCENT') THEN 'Lucent'
            WHEN REGEXP_CONTAINS(UPPER(owner), r'ALCATEL|LUCENT') THEN 'Alcatel and Alcatel-Lucent'
            WHEN UPPER(owner) LIKE '%NOKIA%' THEN 'Nokia'
            ELSE 'Other'
        END AS origin
    FROM us
), per_patent AS (
    SELECT publication_number,
        MAX(origin = 'Alcatel and Alcatel-Lucent') AS has_alcatel_group,
        MAX(origin = 'Lucent') AS has_lucent
    FROM c
    GROUP BY publication_number
)
SELECT
    COUNT(*) AS unique_patents,
    COUNTIF(has_alcatel_group) AS alcatel_group_patents,
    COUNTIF(has_lucent) AS lucent_patents,
    COUNTIF(has_alcatel_group OR has_lucent) AS alcatel_lucent_union,
    COUNTIF(has_alcatel_group AND has_lucent) AS overlap_double_counted,
    ROUND(COUNTIF(has_alcatel_group OR has_lucent) / COUNT(*) * 100, 1) AS alcatel_lucent_share_pct
FROM per_patent;


## `20_category_period.sql`

Raw retrieval-pool family counts by company and priority period, for H2.


In [ ]:
-- Raw retrieval-pool family counts by company and priority period, for H2.
-- Same company attribution and filters as 16_codebook_sample.sql (US granted members since Nov 2006).
-- These are POOL counts, not validated class counts: multiply by the per-company precision
-- factors in data/validation/results_precision.csv before comparing, and report the 95% ranges.
-- A family can sit in more than one pool; that is intended.
WITH b AS (
    SELECT 'Nokia' AS co, family_id, cpc_codes, priority_date
    FROM nokia_analytics.portfolio_30y
    WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')
    UNION ALL
    SELECT CASE
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CIENA') THEN 'Ciena'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'CISCO|ACACIA|LUXTERA') THEN 'Cisco'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'ARISTA') THEN 'Arista'
        WHEN REGEXP_CONTAINS(UPPER(ARRAY_TO_STRING(owners, ' ')), r'HUAWEI') THEN 'Huawei'
        ELSE 'Other'
    END, family_id, cpc_codes, priority_date
    FROM nokia_analytics.rivals_30y
    WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')
), fam AS (
    SELECT co, family_id, MIN(priority_date) AS first_priority,
        MAX(EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE REGEXP_CONTAINS(c, r'^(H04B10/|H04J14/|H04Q11/00)'))) AS opt,
        MAX(EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE STARTS_WITH(c, 'H04L45/'))) AS ipr,
        MAX(EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE STARTS_WITH(c, 'H04L49/'))) AS dcs
    FROM b
    WHERE co != 'Other'
    GROUP BY co, family_id
), labelled AS (
    SELECT co, family_id,
        CASE
            WHEN first_priority <= 20101231 THEN 'to 2010'
            WHEN first_priority <= 20151231 THEN '2011-2015'
            WHEN first_priority <= 20201231 THEN '2016-2020'
            ELSE '2021 on'
        END AS period,
        opt, ipr, dcs
    FROM fam
)
SELECT pool, co, period, COUNT(*) AS pool_families
FROM labelled, UNNEST(['OPT', 'IPR', 'DCS']) AS pool
WHERE (pool = 'OPT' AND opt) OR (pool = 'IPR' AND ipr) OR (pool = 'DCS' AND dcs)
GROUP BY pool, co, period
ORDER BY pool, co, period;


## `21_expiry_curve.sql`

Expiry curve for H3: families with grant evidence reaching the 20-year approximation each year.


In [ ]:
-- Expiry curve for H3: families with grant evidence reaching the 20-year approximation each year.
-- Runs on nokia_analytics.family_status (built by 09_family_status.sql).
-- Limits: filing date + 20 years is an approximation (term adjustments, terminal disclaimers and
-- lapses are not visible here); core-owner names only; category split needs the validated
-- classifier applied to all families, which does not exist yet, so this is the overall curve.
SELECT
    CAST(SUBSTR(CAST(first_filing AS STRING), 1, 4) AS INT64) + 20 AS expiry_year,
    COUNT(*) AS families_reaching_20y,
    SUM(COUNT(*)) OVER (ORDER BY CAST(SUBSTR(CAST(first_filing AS STRING), 1, 4) AS INT64)) AS cumulative
FROM nokia_analytics.family_status
WHERE core_owner AND ever_granted
GROUP BY expiry_year
ORDER BY expiry_year;


## `22_infinera_optical.sql`

Infinera split for H1: US-granted families (filed since Nov 2006) with at least one


In [ ]:
-- Infinera split for H1: US-granted families (filed since Nov 2006) with at least one
-- Infinera owner name, against the optical retrieval pool (H04B10/H04J14/H04Q11/00).
-- Pool membership is not a validated optical label; apply the Nokia OPT precision factor
-- from data/validation/results_precision.csv before quoting a class estimate.
WITH fam AS (
    SELECT family_id,
        MAX(EXISTS (SELECT 1 FROM UNNEST(owners) AS o WHERE UPPER(o) LIKE '%INFINERA%')) AS infinera,
        MAX(EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c WHERE REGEXP_CONTAINS(c, r'^(H04B10/|H04J14/|H04Q11/00)'))) AS optical_pool,
        MIN(priority_date) AS first_priority
    FROM nokia_analytics.portfolio_30y
    WHERE country_code = 'US' AND grant_date > 0 AND filing_date >= 20061101 AND STARTS_WITH(kind_code, 'B')
    GROUP BY family_id
)
SELECT
    infinera,
    COUNT(*) AS families,
    COUNTIF(optical_pool) AS optical_pool_families,
    COUNTIF(optical_pool AND first_priority >= 20160101) AS optical_pool_since_2016
FROM fam
GROUP BY infinera
ORDER BY infinera DESC;


## `23_satcom_actor_names.sql`

Name census for the six satcom actors: every assignee name variant matching the search patterns, before any counting. Basis of the name review.

In [ ]:
-- Name census for the sovereign-satcom actors (question log Q24).
-- Broad patterns first, reviewed by eye afterwards — the same procedure as 01 and 02.
-- Deliberately no technology, office or grant filter: those choices belong to the peers table build.
-- One scan; a name matching two patterns lands in the first matching group, which the review will catch.
SELECT
  CASE
    WHEN UPPER(a.name) LIKE '%ICEYE%' THEN 'ICEYE'
    WHEN UPPER(a.name) LIKE '%EUTELSAT%' OR UPPER(a.name) LIKE '%ONEWEB%'
      OR UPPER(a.name) LIKE '%ONE WEB%' OR UPPER(a.name) LIKE '%WORLDVU%' THEN 'Eutelsat/OneWeb'
    WHEN UPPER(a.name) LIKE '%VIASAT%' OR UPPER(a.name) LIKE '%VIA SAT%'
      OR UPPER(a.name) LIKE '%INMARSAT%' THEN 'Viasat'
    WHEN UPPER(a.name) LIKE '%THALES%' OR UPPER(a.name) LIKE '%ALENIA%' THEN 'Thales Alenia Space'
    WHEN UPPER(a.name) LIKE '%AIRBUS%' OR UPPER(a.name) LIKE '%ASTRIUM%'
      OR UPPER(a.name) LIKE '%EADS%' THEN 'Airbus Defence and Space'
  END AS actor,
  a.name AS name,
  COUNT(DISTINCT p.publication_number) AS publications
FROM `patents-public-data.patents.publications` AS p,
  UNNEST(p.assignee_harmonized) AS a
WHERE REGEXP_CONTAINS(UPPER(a.name), r'ICEYE|EUTELSAT|ONEWEB|ONE WEB|WORLDVU|VIASAT|VIA SAT|INMARSAT|THALES|ALENIA|AIRBUS|ASTRIUM|EADS')
GROUP BY actor, name
ORDER BY actor, publications DESC;


## `24_satcom_peers_30y.sql`

Peers table: the five satcom actors' filings since 1996, matched by the exact reviewed names from the census. **Generated file** (~108 KB, 854 exact-name WHEN clauses) — regenerate with `python3 src/review_satcom_names.py` after any change to the name review. The full text is in `sql/24_satcom_peers_30y.sql`; shown here in abbreviated form.

In [ ]:
-- Peers table: satcom actors, filings since 1996 (mirrors 15_rivals_30y.sql).
-- Generated by src/review_satcom_names.py from data/interim/satcom_name_review.csv.
-- Only names marked "keep" in the review are included; "review" rows wait for a decision.
-- Name matching is exact (post-census), not regex: the census is what makes this safe.
CREATE OR REPLACE TABLE nokia_analytics.satcom_peers_30y AS
SELECT
  p.publication_number,
  p.application_number,
  p.family_id,
  p.country_code,
  p.kind_code,
  p.priority_date,
  p.filing_date,
  p.grant_date,
  ARRAY(
    SELECT DISTINCT CASE UPPER(a.name)
      WHEN 'A 3 BY AIRBUS LLC' THEN 'Airbus'
      WHEN 'A CIRCUMFLEX OVER 3 BY AIRBUS LLC' THEN 'Airbus'
      WHEN 'A3 BY AIRBUS LLC' THEN 'Airbus'
      WHEN 'AAIRBUS DEFENCE AND SPACE SAS' THEN 'Airbus'
      -- ... 854 exact-name WHEN clauses and IN lists omitted here ...
      -- full text: sql/24_satcom_peers_30y.sql (generated)


## `25_ukraine_satcom_assignees.sql`

Discovery: Ukrainian assignees holding satcom-pool patents. Result: no corporate Ukrainian satcom patent base.

In [ ]:
-- Ukrainian assignees holding satcom-pool patents (discovery query).
-- Who inside Ukraine holds patents relevant to a sovereign satcom system?
-- CPC prefixes come from the candidate pools in docs/01-research-design/satcom-code-pools.md
-- (broadened to prefix level; this is discovery, not the validated pool census).
SELECT
  a.name,
  COUNT(DISTINCT p.publication_number) AS publications,
  COUNT(DISTINCT p.family_id) AS families
FROM `patents-public-data.patents.publications` AS p, UNNEST(p.assignee_harmonized) AS a
WHERE a.country_code = 'UA'
  AND EXISTS (
    SELECT 1 FROM UNNEST(p.cpc) AS c
    WHERE STARTS_WITH(c.code, 'H04B7/185')
       OR STARTS_WITH(c.code, 'H04B7/195')
       OR STARTS_WITH(c.code, 'H04B10/118')
       OR STARTS_WITH(c.code, 'H04W84/06')
       OR STARTS_WITH(c.code, 'H04K1/')
       OR STARTS_WITH(c.code, 'H04K3/')
  )
GROUP BY a.name
ORDER BY families DESC, publications DESC


## `26_ukraine_office_satcom_filings.sql`

Discovery: satcom-pool patents filed at the Ukrainian office, any owner. Result: 10 owners, 16 families, ever.

In [ ]:
-- Satcom-pool patents filed at the Ukrainian patent office, any owner (discovery query).
-- Which companies consider the Ukrainian market worth protecting satcom technology in?
-- Same CPC prefixes as 25_ukraine_satcom_assignees.sql.
SELECT
  a.name,
  a.country_code AS assignee_country,
  COUNT(DISTINCT p.publication_number) AS ua_filings,
  COUNT(DISTINCT p.family_id) AS families
FROM `patents-public-data.patents.publications` AS p, UNNEST(p.assignee_harmonized) AS a
WHERE p.country_code = 'UA'
  AND EXISTS (
    SELECT 1 FROM UNNEST(p.cpc) AS c
    WHERE STARTS_WITH(c.code, 'H04B7/185')
       OR STARTS_WITH(c.code, 'H04B7/195')
       OR STARTS_WITH(c.code, 'H04B10/118')
       OR STARTS_WITH(c.code, 'H04W84/06')
       OR STARTS_WITH(c.code, 'H04K1/')
       OR STARTS_WITH(c.code, 'H04K3/')
  )
GROUP BY a.name, assignee_country
ORDER BY ua_filings DESC
LIMIT 200


## `27_ukraine_coverage_check.sql`

Coverage check behind query 25: 336,474 UA-office and 239,568 UA-assignee publications exist, so the empty satcom result is real.

In [ ]:
-- Coverage check behind query 25's empty Ukraine result: is the dataset's
-- Ukrainian coverage real, or is "no satcom patents" a data hole?
-- Answer (10 Nov 2026): 336,474 UA-office and 239,568 UA-assignee publications.
SELECT
  (SELECT COUNT(*) FROM `patents-public-data.patents.publications` WHERE country_code = 'UA') AS ua_office_pubs,
  (SELECT COUNT(*) FROM `patents-public-data.patents.publications` AS p
   WHERE EXISTS (SELECT 1 FROM UNNEST(p.assignee_harmonized) AS a WHERE a.country_code = 'UA')) AS ua_assignee_pubs;


## `28_ukraine_entity_check.sql`

Known-entity check for Ukraine's space-industry names. Found ECHOSTAR UKRAINE LLC; no KB Pivdenne/Pivdenmash under these spellings.

In [ ]:
-- Known-entity check: do Ukraine's space-industry names appear in the dataset
-- at all, and is there any large Ukrainian holder we missed?
-- Found ECHOSTAR UKRAINE LLC (114 publications, UA) — followed up in query 29.
-- No KB Pivdenne / Pivdenmash under these spellings; its heritage is rocket
-- platforms (B64G), outside the communications scope by design.
SELECT
  a.name,
  a.country_code,
  COUNT(*) AS pubs
FROM `patents-public-data.patents.publications` AS p, UNNEST(p.assignee_harmonized) AS a
WHERE REGEXP_CONTAINS(UPPER(a.name), r'PIVDEN|YUZH|HARTRON|KHARTRON|KB YUZH|UKRSAT|UKRAINE')
GROUP BY a.name, a.country_code
ORDER BY pubs DESC
LIMIT 50;


## `29_echostar_ukraine_cpc.sql`

Follow-up: EchoStar Ukraine's patents are set-top-box video R&D (H04N), not satellite communications.

In [ ]:
-- Follow-up to query 28: what does EchoStar's Ukrainian entity patent?
-- Answer: set-top-box video R&D (93 of 114 publications in H04N), not
-- satellite communications — correctly outside the satcom pools.
SELECT
  SUBSTR(c.code, 1, 4) AS cpc_class,
  COUNT(DISTINCT p.publication_number) AS pubs
FROM `patents-public-data.patents.publications` AS p,
     UNNEST(p.assignee_harmonized) AS a,
     UNNEST(p.cpc) AS c
WHERE a.name = 'ECHOSTAR UKRAINE LLC'
GROUP BY cpc_class
ORDER BY pubs DESC
LIMIT 10;


## `30_spacex_name_counts.sql`

Does SpaceX patent? 366 publications / 110 families — a trade-secret strategy, not a technology gap.

In [ ]:
-- Does SpaceX patent? Name census for SpaceX/Starlink.
-- Answer (10 Nov 2026): SPACE EXPLORATION TECH CORP has 366 publications /
-- 110 families — about 175x fewer than Airbus, 7x fewer than Viasat.
-- A trade-secret strategy, not a technology gap; recorded in
-- docs/01-research-design/satcom-code-pools.md (known risks).
SELECT
  a.name,
  a.country_code,
  COUNT(*) AS pubs,
  COUNT(DISTINCT p.family_id) AS families
FROM `patents-public-data.patents.publications` AS p, UNNEST(p.assignee_harmonized) AS a
WHERE REGEXP_CONTAINS(UPPER(a.name), r'SPACEX|SPACE EXPLORATION TECH|STARLINK')
GROUP BY a.name, a.country_code
ORDER BY pubs DESC
LIMIT 30;


## `31_iceye_recent_examples.sql`

Recent ICEYE publications from the peers table (query 24), for the claims-reading rehearsal in the validation docs.

In [ ]:
-- Recent ICEYE publications from the peers table, for the claims-reading
-- rehearsal in the validation docs (WO2025252651A1, the SAR antenna patent).
-- Runs on the sandbox table nokia_analytics.satcom_peers_30y (query 24).
SELECT
  publication_number,
  filing_date,
  title_en,
  (SELECT STRING_AGG(code) FROM UNNEST(cpc_codes) AS code) AS cpc
FROM nokia_analytics.satcom_peers_30y
WHERE 'ICEYE' IN UNNEST(owners)
ORDER BY filing_date DESC
LIMIT 8;


## `32_pool_census.sql`

Pool census (validation stage 0): families per satcom pool x owner x period, Nokia lineage included. PRE-VALIDATION counts — the pools are candidates.

In [ ]:
-- Pool census (validation stage 0): families per satcom pool x owner x period.
-- Nokia lineage from portfolio_30y, the five actors from satcom_peers_30y.
-- PRE-VALIDATION discovery counts: the pools are candidates, so these numbers
-- describe what the code prefixes catch, not yet validated capabilities.
-- Pool rules from docs/01-research-design/satcom-code-pools.md.
-- Prefix note: H04B7/1851 (SAT) also prefixes 7/18517 (GRD earth stations),
-- so SAT excludes the 7/18517 branch explicitly.
WITH tagged AS (
  SELECT family_id, 'Nokia (lineage)' AS owner, cpc_codes, priority_date
  FROM nokia_analytics.portfolio_30y
  UNION ALL
  SELECT p.family_id, o AS owner, p.cpc_codes, p.priority_date
  FROM nokia_analytics.satcom_peers_30y AS p, UNNEST(p.owners) AS o
),
memberships AS (
  SELECT family_id, owner, 'SAT' AS pool, priority_date FROM tagged
  WHERE EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c
                WHERE STARTS_WITH(c, 'H04B7/195')
                   OR (STARTS_WITH(c, 'H04B7/1851') AND NOT STARTS_WITH(c, 'H04B7/18517')))
  UNION ALL
  SELECT family_id, owner, 'ISL', priority_date FROM tagged
  WHERE EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c
                WHERE STARTS_WITH(c, 'H04B7/18521') OR STARTS_WITH(c, 'H04B10/118'))
  UNION ALL
  SELECT family_id, owner, 'GRD', priority_date FROM tagged
  WHERE EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c
                WHERE STARTS_WITH(c, 'H04B7/18517') OR STARTS_WITH(c, 'H04B7/18528'))
  UNION ALL
  SELECT family_id, owner, 'NET', priority_date FROM tagged
  WHERE EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c
                WHERE STARTS_WITH(c, 'H04W84/06') OR STARTS_WITH(c, 'H04B7/18584')
                   OR STARTS_WITH(c, 'H04B7/18541'))
  UNION ALL
  SELECT family_id, owner, 'SEC', priority_date FROM tagged
  WHERE EXISTS (SELECT 1 FROM UNNEST(cpc_codes) AS c
                WHERE STARTS_WITH(c, 'H04K1/') OR STARTS_WITH(c, 'H04K3/')
                   OR STARTS_WITH(c, 'H04B7/18565') OR STARTS_WITH(c, 'H04B7/18593'))
)
SELECT
  owner,
  pool,
  CASE
    WHEN priority_date >= 20200101 THEN '2020s'
    WHEN priority_date >= 20100101 THEN '2010s'
    WHEN priority_date >= 20000101 THEN '2000s'
    ELSE 'pre-2000'
  END AS period,
  COUNT(DISTINCT family_id) AS families
FROM memberships
GROUP BY owner, pool, period
ORDER BY owner, pool, period;


## `33_pilot_sample.sql`

Pilot sample (validation stage 1): 30 families per satcom pool, stratified by owner and priority period (largest remainder), deterministic ordering. One US/WO/EP/GB representative publication per family so the numbered sentences at the back are readable in English. Output feeds the blind labelling sheet.

In [ ]:
-- Pilot sample (validation stage 1): 30 families per satcom pool, 150 total.
-- Stratified: proportional allocation across the six owners, then across
-- priority periods inside each owner cell (largest remainder both times).
-- Deterministic: FARM_FINGERPRINT ordering, no RAND().
-- One representative publication per family, restricted to US/WO/EP/GB members
-- so the numbered sentences at the back are readable in English.
-- matched_codes lists every pool prefix that caught the family, so precision
-- can later be reported per code, not only per pool.
-- first_code is the first CPC entry in the representative publication's array,
-- used as a proxy for listing order (the array does not document its order).
WITH tagged AS (
  SELECT family_id, 'Nokia (lineage)' AS owner, publication_number, country_code,
         kind_code, cpc_codes, priority_date, title_en
  FROM nokia_analytics.portfolio_30y
  UNION ALL
  SELECT p.family_id, o AS owner, p.publication_number, p.country_code,
         p.kind_code, p.cpc_codes, p.priority_date, p.title_en
  FROM nokia_analytics.satcom_peers_30y AS p, UNNEST(p.owners) AS o
),
memberships AS (
  SELECT family_id, owner, 'SAT' AS pool, c AS matched_code FROM tagged, UNNEST(cpc_codes) AS c
  WHERE STARTS_WITH(c, 'H04B7/195')
     OR (STARTS_WITH(c, 'H04B7/1851') AND NOT STARTS_WITH(c, 'H04B7/18517'))
  UNION ALL
  SELECT family_id, owner, 'ISL', c FROM tagged, UNNEST(cpc_codes) AS c
  WHERE STARTS_WITH(c, 'H04B7/18521') OR STARTS_WITH(c, 'H04B10/118')
  UNION ALL
  SELECT family_id, owner, 'GRD', c FROM tagged, UNNEST(cpc_codes) AS c
  WHERE STARTS_WITH(c, 'H04B7/18517') OR STARTS_WITH(c, 'H04B7/18528')
  UNION ALL
  SELECT family_id, owner, 'NET', c FROM tagged, UNNEST(cpc_codes) AS c
  WHERE STARTS_WITH(c, 'H04W84/06') OR STARTS_WITH(c, 'H04B7/18584')
     OR STARTS_WITH(c, 'H04B7/18541')
  UNION ALL
  SELECT family_id, owner, 'SEC', c FROM tagged, UNNEST(cpc_codes) AS c
  WHERE STARTS_WITH(c, 'H04K1/') OR STARTS_WITH(c, 'H04K3/')
     OR STARTS_WITH(c, 'H04B7/18565') OR STARTS_WITH(c, 'H04B7/18593')
),
fams AS (
  SELECT
    m.family_id,
    m.owner,
    m.pool,
    STRING_AGG(DISTINCT m.matched_code ORDER BY m.matched_code) AS matched_codes,
    MIN(t.priority_date) AS priority_date
  FROM memberships AS m
  JOIN (
    SELECT family_id, owner, MIN(priority_date) AS priority_date
    FROM tagged
    GROUP BY family_id, owner
  ) AS t
    ON t.family_id = m.family_id AND t.owner = m.owner
  JOIN (
    SELECT DISTINCT family_id, owner
    FROM tagged
    WHERE country_code IN ('US', 'WO', 'EP', 'GB')
  ) AS e
    ON e.family_id = m.family_id AND e.owner = m.owner
  GROUP BY m.family_id, m.owner, m.pool
),
reps AS (
  SELECT family_id, owner, publication_number, title_en, cpc_codes
  FROM (
    SELECT t.*,
      ROW_NUMBER() OVER (
        PARTITION BY t.family_id, t.owner
        ORDER BY CASE
          WHEN t.country_code = 'US' AND STARTS_WITH(t.kind_code, 'B') THEN 1
          WHEN t.country_code = 'US' THEN 2
          WHEN t.country_code = 'WO' THEN 3
          WHEN t.country_code = 'EP' THEN 4
          ELSE 5
        END, t.publication_number
      ) AS pick
    FROM tagged AS t
    WHERE t.country_code IN ('US', 'WO', 'EP', 'GB')
  )
  WHERE pick = 1
),
pool_fams AS (
  SELECT
    f.family_id, f.owner, f.pool, f.matched_codes, f.priority_date,
    CASE
      WHEN f.priority_date >= 20200101 THEN '2020s'
      WHEN f.priority_date >= 20100101 THEN '2010s'
      WHEN f.priority_date >= 20000101 THEN '2000s'
      ELSE 'pre-2000'
    END AS period,
    r.publication_number, r.title_en,
    r.cpc_codes[SAFE_OFFSET(0)] AS first_code
  FROM fams AS f
  JOIN reps AS r ON r.family_id = f.family_id AND r.owner = f.owner
),
owner_avail AS (
  SELECT pool, owner, COUNT(*) AS avail
  FROM pool_fams
  GROUP BY pool, owner
),
owner_share AS (
  SELECT
    pool, owner, avail,
    30.0 * avail / SUM(avail) OVER (PARTITION BY pool) AS share
  FROM owner_avail
),
owner_base AS (
  SELECT
    pool, owner, avail,
    LEAST(CAST(FLOOR(share) AS INT64), avail) AS base,
    share - FLOOR(share) AS frac
  FROM owner_share
),
owner_quota AS (
  SELECT
    b.pool, b.owner,
    b.base + IF(b.base < b.avail AND r.rn <= 30 - s.assigned, 1, 0) AS quota
  FROM owner_base AS b
  CROSS JOIN (SELECT pool, SUM(base) AS assigned FROM owner_base GROUP BY pool) AS s
  LEFT JOIN (
    SELECT pool, owner,
      ROW_NUMBER() OVER (PARTITION BY pool ORDER BY frac DESC, owner) AS rn
    FROM owner_base
    WHERE base < avail
  ) AS r ON r.pool = b.pool AND r.owner = b.owner
  WHERE s.pool = b.pool
),
period_avail AS (
  SELECT pool, owner, period, COUNT(*) AS avail
  FROM pool_fams
  GROUP BY pool, owner, period
),
period_share AS (
  SELECT
    p.pool, p.owner, p.period, p.avail,
    q.quota * p.avail / SUM(p.avail) OVER (PARTITION BY p.pool, p.owner) AS share,
    q.quota
  FROM period_avail AS p
  JOIN owner_quota AS q ON q.pool = p.pool AND q.owner = p.owner
),
period_base AS (
  SELECT
    pool, owner, period, avail, quota,
    LEAST(CAST(FLOOR(share) AS INT64), avail) AS base,
    share - FLOOR(share) AS frac
  FROM period_share
),
period_quota AS (
  SELECT
    b.pool, b.owner, b.period,
    b.base + IF(b.base < b.avail AND r.rn <= b.quota - s.assigned, 1, 0) AS quota
  FROM period_base AS b
  CROSS JOIN (
    SELECT pool, owner, SUM(base) AS assigned
    FROM period_base GROUP BY pool, owner
  ) AS s
  LEFT JOIN (
    SELECT pool, owner, period,
      ROW_NUMBER() OVER (PARTITION BY pool, owner ORDER BY frac DESC, period) AS rn
    FROM period_base
    WHERE base < avail
  ) AS r ON r.pool = b.pool AND r.owner = b.owner AND r.period = b.period
  WHERE s.pool = b.pool AND s.owner = b.owner
),
numbered AS (
  SELECT
    f.*,
    q.quota,
    ROW_NUMBER() OVER (
      PARTITION BY f.pool, f.owner, f.period
      ORDER BY FARM_FINGERPRINT(CONCAT(CAST(f.family_id AS STRING), '|', f.pool, '|satcom-pilot'))
    ) AS rn
  FROM pool_fams AS f
  JOIN period_quota AS q ON q.pool = f.pool AND q.owner = f.owner AND q.period = f.period
)
SELECT
  'satcom-pilot' AS stage,
  pool,
  owner AS company,
  publication_number,
  family_id,
  priority_date,
  period,
  matched_codes,
  first_code,
  CASE
    WHEN pool = 'SAT' THEN STARTS_WITH(first_code, 'H04B7/195')
      OR (STARTS_WITH(first_code, 'H04B7/1851') AND NOT STARTS_WITH(first_code, 'H04B7/18517'))
    WHEN pool = 'ISL' THEN STARTS_WITH(first_code, 'H04B7/18521') OR STARTS_WITH(first_code, 'H04B10/118')
    WHEN pool = 'GRD' THEN STARTS_WITH(first_code, 'H04B7/18517') OR STARTS_WITH(first_code, 'H04B7/18528')
    WHEN pool = 'NET' THEN STARTS_WITH(first_code, 'H04W84/06') OR STARTS_WITH(first_code, 'H04B7/18584')
      OR STARTS_WITH(first_code, 'H04B7/18541')
    WHEN pool = 'SEC' THEN STARTS_WITH(first_code, 'H04K1/') OR STARTS_WITH(first_code, 'H04K3/')
      OR STARTS_WITH(first_code, 'H04B7/18565') OR STARTS_WITH(first_code, 'H04B7/18593')
  END AS first_code_matches_pool,
  title_en
FROM numbered
WHERE rn <= quota
ORDER BY pool, company, period, publication_number;


## Saved outputs and validation

Saved query outputs live under `data/raw/`, `data/processed/`, and `data/validation/`. Run `python3 src/check_outputs.py` after refreshing outputs. Queries 19–22 are written but not yet run; their results are needed for the distinct Alcatel/Lucent share and hypotheses H1–H3.
